In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import uuid

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproyecto")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/citibike/*.csv"

tabla_destino = f"{catalogo}.{esquema}.citibike_trips"

source_system = "CITIBIKE_OFFICIAL"

batch_id = str(uuid.uuid4())

print(f"Ruta origen    : {ruta}")
print(f"Tabla destino  : {tabla_destino}")
print(f"Source system  : {source_system}")
print(f"Batch ID       : {batch_id}")

Ruta origen    : abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/*.csv
Tabla destino  : catalog_au.bronze.citibike_trips
Source system  : CITIBIKE_OFFICIAL
Batch ID       : 55e53873-ada2-4286-b3f6-4d24e53e6e00


In [0]:
df_citibike = spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta)

df_citibike.printSchema()

root
 |-- tripduration: integer (nullable = true)
 |-- starttime: string (nullable = true)
 |-- stoptime: string (nullable = true)
 |-- start station id: integer (nullable = true)
 |-- start station name: string (nullable = true)
 |-- start station latitude: double (nullable = true)
 |-- start station longitude: double (nullable = true)
 |-- end station id: integer (nullable = true)
 |-- end station name: string (nullable = true)
 |-- end station latitude: double (nullable = true)
 |-- end station longitude: double (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth year: double (nullable = true)
 |-- gender: integer (nullable = true)



In [0]:
citibike_schema = StructType(fields=[
    StructField("tripduration", LongType(), True),
    StructField("starttime", TimestampType(), True),
    StructField("stoptime", TimestampType(), True),

    StructField("start station id", IntegerType(), True),
    StructField("start station name", StringType(), True),
    StructField("start station latitude", DoubleType(), True),
    StructField("start station longitude", DoubleType(), True),

    StructField("end station id", IntegerType(), True),
    StructField("end station name", StringType(), True),
    StructField("end station latitude", DoubleType(), True),
    StructField("end station longitude", DoubleType(), True),

    StructField("bikeid", IntegerType(), True),
    StructField("usertype", StringType(), True),
    StructField("birth year", DoubleType(), True),
    StructField("gender", IntegerType(), True)
])

In [0]:
df_citibike_final = spark.read\
    .option('header', True)\
    .option('timestampFormat', 'M/d/yyyy H:mm:ss')\
    .schema(citibike_schema)\
    .csv(ruta)\
    .select(
        "*",
        col("_metadata.file_path").alias("_source_file")
    )

In [0]:
citibike_selected_df = df_citibike_final.select(
    col("tripduration"),
    col("starttime"),
    col("stoptime"),

    col("start station id"),
    col("start station name"),
    col("start station latitude"),
    col("start station longitude"),

    col("end station id"),
    col("end station name"),
    col("end station latitude"),
    col("end station longitude"),

    col("bikeid"),
    col("usertype"),
    col("birth year"),
    col("gender"),

    col("_source_file")
)

In [0]:
citibike_renamed_df = citibike_selected_df\
    .withColumnRenamed("start station id", "start_station_id")\
    .withColumnRenamed("start station name", "start_station_name")\
    .withColumnRenamed("start station latitude", "start_station_latitude")\
    .withColumnRenamed("start station longitude", "start_station_longitude")\
    .withColumnRenamed("end station id", "end_station_id")\
    .withColumnRenamed("end station name", "end_station_name")\
    .withColumnRenamed("end station latitude", "end_station_latitude")\
    .withColumnRenamed("end station longitude", "end_station_longitude")\
    .withColumn("birth_year", col("birth year").cast("int"))\
    .drop("birth year")

In [0]:
citibike_final_df = citibike_renamed_df\
    .withColumn("_ingestion_timestamp", current_timestamp())\
    .withColumn("_source_system", lit(source_system))\
    .withColumn("_batch_id", lit(batch_id))

citibike_final_df = citibike_final_df.select(
    col("tripduration"),
    col("starttime"),
    col("stoptime"),

    col("start_station_id"),
    col("start_station_name"),
    col("start_station_latitude"),
    col("start_station_longitude"),

    col("end_station_id"),
    col("end_station_name"),
    col("end_station_latitude"),
    col("end_station_longitude"),

    col("bikeid"),
    col("usertype"),
    col("birth_year"),
    col("gender"),

    col("_ingestion_timestamp"),
    col("_source_file"),
    col("_source_system"),
    col("_batch_id")
)

In [0]:
citibike_final_df.select(
    count("*").alias("total_registros"),
    count("starttime").alias("starttime_validos"),
    count("stoptime").alias("stoptime_validos"),
    count("birth_year").alias("birth_year_validos")
).show()

+---------------+-----------------+----------------+------------------+
|total_registros|starttime_validos|stoptime_validos|birth_year_validos|
+---------------+-----------------+----------------+------------------+
|        5676020|          5676020|         5676020|           5026409|
+---------------+-----------------+----------------+------------------+



In [0]:
citibike_final_df.filter(
    col("birth_year").isNull()
).count()

649611

In [0]:
citibike_final_df.write.mode("overwrite")\
    .insertInto(tabla_destino)

In [0]:
total_raw = df_citibike_final.count()
total_bronze = spark.table(tabla_destino).count()

print(f"Registros RAW    : {total_raw:,}")
print(f"Registros BRONZE : {total_bronze:,}")

if total_raw != total_bronze:
    raise Exception(
        "La cantidad de registros RAW y BRONZE no coincide."
    )

print("Ingesta citibike_trips finalizada correctamente.")

Registros RAW    : 5,676,020
Registros BRONZE : 5,676,020
Ingesta citibike_trips finalizada correctamente.


In [0]:
display(
    spark.table(tabla_destino).limit(10)
)

tripduration,starttime,stoptime,start_station_id,start_station_name,start_station_latitude,start_station_longitude,end_station_id,end_station_name,end_station_latitude,end_station_longitude,bikeid,usertype,birth_year,gender,_ingestion_timestamp,_source_file,_source_system,_batch_id
461,2016-02-01T00:00:08Z,2016-02-01T00:07:49Z,480,W 53 St & 10 Ave,40.76669671,-73.99061728,524,W 43 St & 6 Ave,40.75527307,-73.98316936,23292,Subscriber,1966,1,2026-08-13T18:00:22.824531Z,abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201602-citibike-tripdata_1.csv,CITIBIKE_OFFICIAL,55e53873-ada2-4286-b3f6-4d24e53e6e00
297,2016-02-01T00:00:56Z,2016-02-01T00:05:53Z,463,9 Ave & W 16 St,40.74206539,-74.00443172,380,W 4 St & 7 Ave S,40.73401143,-74.00293877,15329,Subscriber,1977,1,2026-08-13T18:00:22.824531Z,abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201602-citibike-tripdata_1.csv,CITIBIKE_OFFICIAL,55e53873-ada2-4286-b3f6-4d24e53e6e00
280,2016-02-01T00:01:00Z,2016-02-01T00:05:40Z,3134,3 Ave & E 62 St,40.76312584,-73.96526895,3141,1 Ave & E 68 St,40.76500525,-73.95818491,22927,Subscriber,1987,1,2026-08-13T18:00:22.824531Z,abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201602-citibike-tripdata_1.csv,CITIBIKE_OFFICIAL,55e53873-ada2-4286-b3f6-4d24e53e6e00
662,2016-02-01T00:01:00Z,2016-02-01T00:12:02Z,537,Lexington Ave & E 24 St,40.74025878,-73.98409214,428,E 3 St & 1 Ave,40.72467721,-73.98783413,20903,Subscriber,1983,2,2026-08-13T18:00:22.824531Z,abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201602-citibike-tripdata_1.csv,CITIBIKE_OFFICIAL,55e53873-ada2-4286-b3f6-4d24e53e6e00
355,2016-02-01T00:01:41Z,2016-02-01T00:07:36Z,284,Greenwich Ave & 8 Ave,40.7390169121,-74.0026376103,521,8 Ave & W 31 St,40.75096734871598,-73.99444207549095,23228,Subscriber,1978,1,2026-08-13T18:00:22.824531Z,abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201602-citibike-tripdata_1.csv,CITIBIKE_OFFICIAL,55e53873-ada2-4286-b3f6-4d24e53e6e00
1034,2016-02-01T00:01:44Z,2016-02-01T00:18:58Z,223,W 13 St & 7 Ave,40.73781509,-73.99994661,305,E 58 St & 3 Ave,40.76095756,-73.96724467,23761,Subscriber,1966,1,2026-08-13T18:00:22.824531Z,abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201602-citibike-tripdata_1.csv,CITIBIKE_OFFICIAL,55e53873-ada2-4286-b3f6-4d24e53e6e00
601,2016-02-01T00:02:15Z,2016-02-01T00:12:17Z,525,W 34 St & 11 Ave,40.75594159,-74.0021163,472,E 32 St & Park Ave,40.7457121,-73.98194829,22934,Subscriber,1990,1,2026-08-13T18:00:22.824531Z,abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201602-citibike-tripdata_1.csv,CITIBIKE_OFFICIAL,55e53873-ada2-4286-b3f6-4d24e53e6e00
1615,2016-02-01T00:02:45Z,2016-02-01T00:29:40Z,490,8 Ave & W 33 St,40.751551,-73.993934,473,Rivington St & Chrystie St,40.72110063,-73.9919254,15342,Subscriber,1966,1,2026-08-13T18:00:22.824531Z,abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201602-citibike-tripdata_1.csv,CITIBIKE_OFFICIAL,55e53873-ada2-4286-b3f6-4d24e53e6e00
614,2016-02-01T00:02:56Z,2016-02-01T00:13:10Z,459,W 20 St & 11 Ave,40.746745,-74.007756,237,E 11 St & 2 Ave,40.73047309,-73.98672378,15883,Subscriber,1991,1,2026-08-13T18:00:22.824531Z,abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201602-citibike-tripdata_1.csv,CITIBIKE_OFFICIAL,55e53873-ada2-4286-b3f6-4d24e53e6e00
268,2016-02-01T00:02:56Z,2016-02-01T00:07:25Z,345,W 13 St & 6 Ave,40.73649403,-73.99704374,285,Broadway & E 14 St,40.73454567,-73.99074142,16831,Subscriber,1969,2,2026-08-13T18:00:22.824531Z,abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201602-citibike-tripdata_1.csv,CITIBIKE_OFFICIAL,55e53873-ada2-4286-b3f6-4d24e53e6e00


In [0]:
display(
    df_citibike_final
    .select("_source_file")
    .distinct()
    .orderBy("_source_file")
)

_source_file
abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201601-citibike-tripdata_1.csv
abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201602-citibike-tripdata_1.csv
abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201603-citibike-tripdata_1.csv
abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201604-citibike-tripdata_1.csv
abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201604-citibike-tripdata_2.csv
abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201605-citibike-tripdata_1.csv
abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201605-citibike-tripdata_2.csv
abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201606-citibike-tripdata_1.csv
abfss://raw@adlsproyecto.dfs.core.windows.net/citibike/201606-citibike-tripdata_2.csv


In [0]:
cantidad_archivos = (
    df_citibike_final
    .select("_source_file")
    .distinct()
    .count()
)

print(f"Archivos procesados: {cantidad_archivos}")

if cantidad_archivos != 9:
    raise Exception(
        f"Se esperaban 9 archivos Citi Bike y se encontraron {cantidad_archivos}."
    )

Archivos procesados: 9
